In [ ]:
# Cell 1 — load and validate the saved Day 12 predictions.
from pathlib import Path
import json, time
import numpy as np
import pandas as pd

INPUT = Path('/kaggle/input')
OUT = Path('/kaggle/working/day17-fdr-catalog-size-audit')
OUT.mkdir(parents=True, exist_ok=True)

prediction_paths = list(INPUT.rglob('day12_fdr_test_predictions.csv'))
summary_paths = list(INPUT.rglob('day12_fdr_summary.json'))
if not prediction_paths or not summary_paths:
    raise FileNotFoundError(
        'Attach the successful Cosmic Swan Finale 5 output containing '
        'day12_fdr_test_predictions.csv and day12_fdr_summary.json.'
    )

pred_path = max(prediction_paths, key=lambda p: (p.stat().st_size, str(p)))
summary_path = max(summary_paths, key=lambda p: (p.stat().st_size, str(p)))
day12 = json.loads(summary_path.read_text(encoding='utf-8'))
pred = pd.read_csv(pred_path)
required = {'replicate','q','source_id','is_injected','conformal_pvalue'}
missing = sorted(required.difference(pred.columns))
if missing:
    raise ValueError(f'Day 12 prediction CSV is missing columns: {missing}')
pred['is_injected'] = pred['is_injected'].astype(str).str.strip().str.lower().map({'true':True,'false':False})
if pred['is_injected'].isna().any():
    raise ValueError('Could not parse Day 12 is_injected labels as booleans')
pred['conformal_pvalue'] = pd.to_numeric(pred['conformal_pvalue'], errors='raise')
if not np.isfinite(pred['conformal_pvalue']).all() or ((pred['conformal_pvalue']<0)|(pred['conformal_pvalue']>1)).any():
    raise ValueError('Saved conformal p-values must be finite values in [0, 1]')

# q=.10 rows contain the same pre-BY p-values as q=.05; keep one copy per hypothesis.
base = pred[np.isclose(pd.to_numeric(pred['q']),0.10)].copy()
if base.empty:
    raise ValueError('No q=0.10 rows found in the saved predictions')
base['replicate'] = pd.to_numeric(base['replicate'], errors='raise').astype(int)
base['source_id'] = base['source_id'].astype(str)
if base.duplicated(['replicate','source_id']).any():
    raise ValueError('Expected one q=0.10 p-value per source within each replicate')

expected_reps = int(day12.get('replicates', base['replicate'].nunique()))
catalog_size = int(day12.get('catalog_size', 500))
for rep, group in base.groupby('replicate'):
    if len(group) != catalog_size:
        raise ValueError(f'Replicate {rep} has {len(group)} rows; expected {catalog_size}')
    injected_fraction = float(group['is_injected'].mean())
    if not np.isclose(injected_fraction, float(day12.get('injected_fraction',0.10))):
        raise ValueError(f'Replicate {rep} has unexpected injection fraction {injected_fraction:.3f}')
if base['replicate'].nunique() != expected_reps:
    print(f'[WARNING] Summary expects {expected_reps} replicates; predictions contain {base["replicate"].nunique()}')

calibration_n = int(day12.get('calibration_sources', 0))
minimum_p = 1.0/(calibration_n+1) if calibration_n else float(base['conformal_pvalue'].min())
print('[PASS] Day 12 predictions:', pred_path)
print('[PASS] Day 12 summary:', summary_path)
print(f'[PASS] Replicates={base.replicate.nunique()}, original family size={catalog_size}, '
      f'calibration sources={calibration_n:,}, nominal p-value floor≈{minimum_p:.7g}')
print('[PASS] Each parent catalog has the declared 10% injection fraction.')


In [ ]:
# Cell 2 — compare BY at prespecified family sizes and save the audit.
import matplotlib.pyplot as plt

def benjamini_yekutieli(pvalues, q):
    p = np.asarray(pvalues, dtype=float)
    if p.ndim != 1 or not np.isfinite(p).all() or ((p<0)|(p>1)).any():
        raise ValueError('p-values must be finite, one-dimensional, and in [0,1]')
    if not 0<q<1: raise ValueError('q must be between 0 and 1')
    m=len(p)
    if m==0: return np.zeros(0,dtype=bool)
    harmonic=np.sum(1.0/np.arange(1,m+1))
    order=np.argsort(p)
    passed=np.flatnonzero(p[order] <= q*np.arange(1,m+1)/(m*harmonic))
    rejected=np.zeros(m,dtype=bool)
    if passed.size: rejected[order[:passed[-1]+1]]=True
    return rejected

CATALOG_SIZES=(50,100,200,catalog_size)
Q_LEVELS=(0.05,0.10)
SUBSETS_PER_PARENT={50:20,100:20,200:20,catalog_size:1}
INJECTED_FRACTION=float(day12.get('injected_fraction',0.10))
if any(m%10 for m in CATALOG_SIZES):
    raise ValueError('Catalog sizes must permit an exact 10% injected fraction in this benchmark.')
if catalog_size not in CATALOG_SIZES:
    raise ValueError(f'Expected a saved Day 12 family size of 500; got {catalog_size}')

subset_rng=np.random.default_rng(20261117)
records=[]
for parent_rep, group in base.groupby('replicate',sort=True):
    injected=group[group['is_injected']].reset_index(drop=True)
    null=group[~group['is_injected']].reset_index(drop=True)
    for m in CATALOG_SIZES:
        n_alt=int(round(m*INJECTED_FRACTION))
        n_null=m-n_alt
        repetitions=SUBSETS_PER_PARENT[m]
        for subset_no in range(repetitions):
            if m==catalog_size:
                subset=group.reset_index(drop=True)
            else:
                alt_idx=subset_rng.choice(len(injected),size=n_alt,replace=False)
                null_idx=subset_rng.choice(len(null),size=n_null,replace=False)
                subset=pd.concat([injected.iloc[alt_idx],null.iloc[null_idx]],ignore_index=True)
            is_alt=subset['is_injected'].to_numpy(dtype=bool)
            pvals=subset['conformal_pvalue'].to_numpy(dtype=float)
            for q in Q_LEVELS:
                reject=benjamini_yekutieli(pvals,q)
                discoveries=int(reject.sum())
                true_positive=int(np.sum(reject & is_alt))
                false_positive=discoveries-true_positive
                n_positive=int(is_alt.sum())
                records.append({
                    'parent_replicate':int(parent_rep),'subset_index':int(subset_no),
                    'catalog_size':int(m),'q_target':float(q),'null_hypotheses':int(n_null),
                    'injected_hypotheses':int(n_positive),'discoveries':discoveries,
                    'false_discoveries':false_positive,'true_injected_detections':true_positive,
                    'fdp':false_positive/max(discoveries,1),
                    'injection_power':true_positive/max(n_positive,1),
                })

subsets=pd.DataFrame(records)
subsets.to_csv(OUT/'day17_fdr_subcatalog_metrics.csv',index=False)

def cluster_bootstrap(values, seed, repetitions=3000):
    values=np.asarray(values,dtype=float)
    rng=np.random.default_rng(seed)
    draws=rng.choice(values,size=(repetitions,len(values)),replace=True).mean(axis=1)
    return [float(np.quantile(draws,0.025)),float(np.quantile(draws,0.975))]

summary_rows=[]
for (m,q),part in subsets.groupby(['catalog_size','q_target'],sort=True):
    # Resample original 500-hypothesis parent catalogs as clusters; their small subsets are dependent.
    parent=part.groupby('parent_replicate',sort=True).agg(
        mean_fdp=('fdp','mean'),mean_power=('injection_power','mean'),
        mean_discoveries=('discoveries','mean'),any_discovery=('discoveries',lambda x:float((x>0).mean()))
    )
    fdr_ci=cluster_bootstrap(parent['mean_fdp'],seed=771+int(m)+int(q*1000))
    power_ci=cluster_bootstrap(parent['mean_power'],seed=1771+int(m)+int(q*1000))
    summary_rows.append({
        'catalog_size':int(m),'q_target':float(q),'parent_replicates':int(len(parent)),
        'subcatalogs_per_parent':int(SUBSETS_PER_PARENT[int(m)]),
        'mean_FDR':float(parent['mean_fdp'].mean()),'FDR_cluster_bootstrap_95_low':fdr_ci[0],
        'FDR_cluster_bootstrap_95_high':fdr_ci[1],
        'mean_injection_power':float(parent['mean_power'].mean()),
        'power_cluster_bootstrap_95_low':power_ci[0],'power_cluster_bootstrap_95_high':power_ci[1],
        'mean_discoveries_per_catalog':float(parent['mean_discoveries'].mean()),
        'fraction_subcatalogs_with_any_discovery':float((part['discoveries']>0).mean()),
        'empirical_mean_FDR_at_or_below_q':bool(parent['mean_fdp'].mean()<=q),
    })
summary_df=pd.DataFrame(summary_rows)
summary_df.to_csv(OUT/'day17_fdr_catalog_size_metrics.csv',index=False)

fig,axes=plt.subplots(1,2,figsize=(12,4.8))
for q,part in summary_df.groupby('q_target'):
    part=part.sort_values('catalog_size')
    axes[0].errorbar(part.catalog_size,part.mean_FDR,
        yerr=[part.mean_FDR-part.FDR_cluster_bootstrap_95_low,
              part.FDR_cluster_bootstrap_95_high-part.mean_FDR],
        marker='o',capsize=3,label=f'q={q:g}')
    axes[1].errorbar(part.catalog_size,part.mean_injection_power,
        yerr=[part.mean_injection_power-part.power_cluster_bootstrap_95_low,
              part.power_cluster_bootstrap_95_high-part.mean_injection_power],
        marker='o',capsize=3,label=f'q={q:g}')
axes[0].plot([min(CATALOG_SIZES),max(CATALOG_SIZES)],[0.05,0.05],'--',color='gray',linewidth=1)
axes[0].plot([min(CATALOG_SIZES),max(CATALOG_SIZES)],[0.10,0.10],'--',color='black',linewidth=1)
axes[0].set(xlabel='Prespecified family size',ylabel='Mean false discovery proportion',
            title='Empirical FDR by family size',ylim=(-0.005,0.12))
axes[1].set(xlabel='Prespecified family size',ylabel='Mean injected-signal power',
            title='Power by family size',ylim=(-0.01,1.0))
for ax in axes: ax.set_xticks(CATALOG_SIZES); ax.legend()
fig.tight_layout(); fig.savefig(OUT/'day17_fdr_size_sensitivity.png',dpi=160); plt.show()

summary={
    'created_utc':pd.Timestamp.now(tz='UTC').isoformat(),
    'source_predictions':str(pred_path),'source_day12_summary':str(summary_path),
    'method':'Reapply the original BY procedure to random, stratified subsets of saved Day 12 controlled catalogs; no model or p-value refitting.',
    'catalog_sizes':list(CATALOG_SIZES),'q_levels':list(Q_LEVELS),
    'subcatalogs_per_parent':{str(k):int(v) for k,v in SUBSETS_PER_PARENT.items()},
    'parent_replicates':int(base['replicate'].nunique()),'injected_fraction':INJECTED_FRACTION,
    'calibration_sources':calibration_n,'nominal_conformal_pvalue_floor':minimum_p,
    'bootstrap_unit':'original Day 12 500-hypothesis replicate; subcatalogs from the same parent stay clustered',
    'summary_rows':summary_df.to_dict(orient='records'),
    'interpretation_limits':[
        'This is a family-size sensitivity analysis of the saved synthetic benchmark, not a new independent validation run.',
        'The number of hypotheses must be declared before inspecting their p-values. Selecting the top-scoring subset from a larger family and applying this result would invalidate the reported BY interpretation.',
        'Injection labels are used only to construct controlled 10%-injection subcatalogs for evaluation.',
        'The FDR scope is detecting synthetic injections on sampled TESS backgrounds, not astrophysical discoveries in the full real TESS catalog.'
    ],
    'outputs':['day17_fdr_catalog_size_metrics.csv','day17_fdr_subcatalog_metrics.csv',
               'day17_fdr_size_sensitivity.png','day17_fdr_size_sensitivity_summary.json',
               'day17_fdr_size_sensitivity_report.md']
}
(OUT/'day17_fdr_size_sensitivity_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')

lines=[
    '# Day 17 FDR catalog-size audit','',
    f"- Reused Day 12's {base['replicate'].nunique()} saved held-out synthetic catalogs; no model retraining or new injections.",
    f'- Calibration set size: {calibration_n:,}; minimum split-conformal p-value: {minimum_p:.7g}.',
    '- Family sizes and q values were varied as an explicit sensitivity analysis. Each smaller catalog retained exactly 10% injected hypotheses.',
    '- Bootstrap intervals resample parent Day 12 catalogs so subsets drawn from the same parent are not treated as independent.',
    '- The interpretation applies only when the tested family size is prespecified before seeing its p-values.',
    '- It does not measure FDR for astrophysical discoveries in the full TESS catalog.','',
    '| Family size | q | Mean FDR | Cluster bootstrap 95% interval | Mean injection power | Mean discoveries |',
    '|---:|---:|---:|:---:|---:|---:|',
]
for r in summary_rows:
    lines.append(f"| {r['catalog_size']} | {r['q_target']:.2f} | {r['mean_FDR']:.4f} | [{r['FDR_cluster_bootstrap_95_low']:.4f}, {r['FDR_cluster_bootstrap_95_high']:.4f}] | {r['mean_injection_power']:.4f} | {r['mean_discoveries_per_catalog']:.3f} |")
(OUT/'day17_fdr_size_sensitivity_report.md').write_text(chr(10).join(lines)+chr(10),encoding='utf-8')
print('Saved Day 17 artifacts under:',OUT)
print(summary_df.to_string(index=False))
print('Scope: synthetic-injection family-size sensitivity only; prespecify family size before seeing p-values.')
